# ASTRA — Data Science & Optimization Pipeline
### End-to-End Benchmark, Training & Validation for East Coast India Maritime Supply Chain

This notebook implements and validates the **4 core analytical components** of ASTRA:
1. **Model 1 (Predictive ML):** Freight Rate Forward Forecaster (*LightGBM Regressor*)
2. **Model 2 (Predictive ML):** Port Waiting Queue Regressor (*GBDT Regressor*)
3. **Model 3 (Classification ML):** Port Congestion & Demurrage Risk Classifier (*Multi-Class GBDT*)
4. **Method 4 (Optimization):** Multimodal Fleet & Truck Logistics Allocation (*Mixed-Integer Linear Programming - MILP*)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
import lightgbm as lgb
from scipy.optimize import milp, LinearConstraint, Bounds
import joblib
print('Data science and optimization libraries loaded successfully!')

## 1. Model 1: Freight Rate Forward Forecaster (LightGBM)

In [ ]:
# Load multi-corridor Baltic Dry historical freight dataset (33,122 rows)
df_freight = pd.read_csv('../datasets/baltic_dry_freight_multicorridor.csv')
df_freight['Date'] = pd.to_datetime(df_freight['Date'])
df_freight = df_freight.sort_values(by=['Origin_Port', 'Destination_Port', 'Vessel_Class', 'Date']).reset_index(drop=True)

# Feature engineering: Lag rates and 7-day rolling indicators
df_freight['Lag_1_Rate'] = df_freight.groupby(['Origin_Port', 'Destination_Port', 'Vessel_Class'])['Spot_Freight_Rate_USD_Per_MT'].shift(1)
df_freight['Lag_7_Rate'] = df_freight.groupby(['Origin_Port', 'Destination_Port', 'Vessel_Class'])['Spot_Freight_Rate_USD_Per_MT'].shift(7)
df_freight['Rolling_7_BDI'] = df_freight.groupby(['Origin_Port', 'Destination_Port', 'Vessel_Class'])['BDI_Index'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_freight['Rolling_7_Bunker'] = df_freight.groupby(['Origin_Port', 'Destination_Port', 'Vessel_Class'])['VLSFO_Bunker_USD_Per_Ton'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_freight = df_freight.dropna().reset_index(drop=True)

cat_cols = ['Origin_Port', 'Destination_Port', 'Vessel_Class']
for col in cat_cols:
    df_freight[col] = df_freight[col].astype('category')

feature_cols = ['BDI_Index', 'VLSFO_Bunker_USD_Per_Ton', 'Brent_Crude_USD_Per_Bbl', 'Monsoon_Wave_Height_M', 'Lag_1_Rate', 'Lag_7_Rate', 'Rolling_7_BDI', 'Rolling_7_Bunker', 'Origin_Port', 'Destination_Port', 'Vessel_Class']
target_col = 'Spot_Freight_Rate_USD_Per_MT'

# Temporal out-of-sample split (4 years train, 1 year forward test)
train_mask = df_freight['Date'] < '2025-01-01'
test_mask = df_freight['Date'] >= '2025-01-01'
X_train, y_train = df_freight.loc[train_mask, feature_cols], df_freight.loc[train_mask, target_col]
X_test, y_test = df_freight.loc[test_mask, feature_cols], df_freight.loc[test_mask, target_col]

# Model training
model1 = lgb.LGBMRegressor(n_estimators=160, learning_rate=0.05, max_depth=5, num_leaves=24, min_child_samples=25, random_state=42, verbose=-1)
model1.fit(X_train, y_train)

y_train_pred = model1.predict(X_train)
y_test_pred = model1.predict(X_test)

print(f"Train R² Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"Test R² Score:  {r2_score(y_test, y_test_pred):.4f}")
print(f"Test MAE:       ${mean_absolute_error(y_test, y_test_pred):.3f} / MT")
print(f"Test RMSE:      ${np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f} / MT")

## 2. Model 2 & 3: Port Waiting Time Regressor & Congestion Risk Classifier

In [ ]:
df_ports = pd.read_csv('../datasets/east_coast_india_port_telemetry.csv')
port_features = ['Max_Draft_M', 'Max_LOA_M', 'Total_Berths', 'Operational_Berths', 'Current_Vessels_In_Queue', 'Cargo_Handling_Capacity_TPD']
X_port = df_ports[port_features]

# Model 2: Waiting Hours Regressor
model2 = GradientBoostingRegressor(n_estimators=40, learning_rate=0.08, max_depth=2, random_state=42)
model2.fit(X_port, df_ports['Historical_Waiting_Hours'])
wait_pred = model2.predict(X_port)
print(f"Model 2 - Port Waiting R² Score: {r2_score(df_ports['Historical_Waiting_Hours'], wait_pred):.4f}")
print(f"Model 2 - Port Waiting MAE:      {mean_absolute_error(df_ports['Historical_Waiting_Hours'], wait_pred):.2f} hours")

# Model 3: Congestion Risk Classifier
model3 = GradientBoostingClassifier(n_estimators=30, learning_rate=0.08, max_depth=2, random_state=42)
model3.fit(X_port, df_ports['Congestion_Risk_Level'])
risk_pred = model3.predict(X_port)
print(f"Model 3 - Risk Classifier Accuracy: {accuracy_score(df_ports['Congestion_Risk_Level'], risk_pred) * 100:.1f}%")
print(f"Model 3 - Risk Classifier F1-Score: {f1_score(df_ports['Congestion_Risk_Level'], risk_pred, average='weighted'):.4f}")

## 3. Method 4: Mixed-Integer Linear Programming (MILP) Fleet & Truck Optimizer

In [ ]:
# Exact MILP Mathematical Solver via SciPy HiGHS
vessels = [
    {'name': 'Handysize', 'dwt': 35000, 'draft': 10.0, 'rate': 22.5, 'fuel': 18.5 * 585 * 14, 'demurrage': 950 * 12},
    {'name': 'Supramax',  'dwt': 58000, 'draft': 12.5, 'rate': 18.4, 'fuel': 24.2 * 585 * 14, 'demurrage': 1100 * 12},
    {'name': 'Panamax',   'dwt': 74000, 'draft': 13.8, 'rate': 16.9, 'fuel': 31.8 * 585 * 14, 'demurrage': 1200 * 12},
    {'name': 'Capesize',  'dwt': 180000, 'draft': 18.2, 'rate': 11.4, 'fuel': 48.5 * 585 * 14, 'demurrage': 1800 * 12}
]
cargo_qty = 70000 # 70k MT
port_max_draft = 14.5 # Paradip Port max draft depth

c_vessel = [v['rate'] * cargo_qty + v['fuel'] + v['demurrage'] for v in vessels]
c_truck = 4.80 * 40 # $4.80/ton * 40T
c = np.array(c_vessel + [c_truck])

# Constraints matrix
A = np.array([
    [1, 1, 1, 1, 0],
    [-35000, -58000, -74000, -180000, 0],
    [10.0, 12.5, 13.8, 18.2, 0],
    [0, 0, 0, 0, -40]
])
b_l = np.array([1, -np.inf, -np.inf, -np.inf])
b_u = np.array([1, -cargo_qty, port_max_draft, -cargo_qty])

constraints = LinearConstraint(A, b_l, b_u)
integrality = np.array([1, 1, 1, 1, 1])
bounds = Bounds(lb=[0, 0, 0, 0, 0], ub=[1, 1, 1, 1, 2000])

res = milp(c=c, integrality=integrality, constraints=constraints, bounds=bounds)
selected_idx = np.argmax(res.x[:4])

print('MILP Solver Result:')
print(f'  Optimal Landed Cost: ${res.fun:,.2f}')
print(f'  Optimal Vessel:      {vessels[selected_idx]["name"]} ({vessels[selected_idx]["dwt"]} DWT, Draft {vessels[selected_idx]["draft"]}m)')
print(f'  Optimal Road Fleet:  {int(res.x[4])} Multi-Axle Trucks (40T capacity)')